# Non-linear PLSR: A Worked Example on the UCI Concrete Compressive Strength Dataset

Partial Least Squares Regression (PLSR) is a natural choice when predictors are numerous
and highly correlated — but its core assumption of a *linear* relationship between
predictors and response can break down in practice. This notebook walks through a concrete
(pun intended) example of when and how non-linear extensions of PLSR help.

We use the **UCI Concrete Compressive Strength dataset** (Yeh, 1998): 1,030 concrete mix
designs described by 8 ingredient/age variables, with the goal of predicting compressive
strength (MPa). The relationship between these ingredients and strength is well known to be
strongly non-linear — UCI's own dataset description notes that "the concrete compressive
strength is a highly nonlinear function of age and ingredients" — making it a good
illustrative case for comparing standard PLSR against a non-linear PLSR variant.

**Structure of this notebook:**
1. Load and explore the data
2. Demonstrate the non-linearity (EDA)
3. Fit standard PLSR as a baseline
4. Fit a non-linear PLSR model and compare performance
5. Key takeaways and practical considerations

In [ ]:
from ucimlrepo import fetch_ucirepo

concrete = fetch_ucirepo(id=165)  # Concrete Compressive Strength
X = concrete.data.features
y = concrete.data.targets

df = X.copy()
df["Strength"] = y

print(df.shape)
df.head()

(1030, 9)


,Cement,Blast Furnace Slag,Fly Ash,Water,Superplasticizer,Coarse Aggregate,Fine Aggregate,Age,Strength
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,79.99
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.89
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,40.27
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365,41.05
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360,44.30


In [ ]:
# Detecting Nonlinearity Before Fitting a Nonlinear PLSR Model

Before reaching for a nonlinear PLSR variant, it is good practice to first show —
not just assume — that the standard linear PLS assumption is actually violated.
Three complementary, well-established diagnostics are used here.

## 1. Latent score vs. response (the PLS-specific diagnostic)

Standard PLSR assumes a **linear inner relation**: the response `y` is a linear
function of the latent scores `t` extracted from the predictors. Plotting the
first latent score `t1` against `y` and comparing it to a smoothed trend
(LOWESS) reveals whether this core assumption holds. Systematic curvature in
this plot is precisely the diagnostic that motivated Wold's original
development of nonlinear PLS (quadratic PLS, spline PLS): if the inner relation
is not a straight line, forcing a linear model onto it discards structure the
model could otherwise capture. This makes it the most *directly relevant*
diagnostic for justifying a non-linear PLSR approach, since it targets the
exact assumption non-linear PLS relaxes.

## 2. Residuals vs. fitted values (standard regression diagnostic)

A textbook diagnostic (Draper & Smith, *Applied Regression Analysis*): under a
correctly specified model, residuals should scatter randomly around zero with
no pattern. A curved band in the residual plot indicates unmodelled structure
— typically a sign of nonlinearity the model has not captured.

## 3. Ramsey's RESET test (formal statistical test)

Ramsey's Regression Equation Specification Error Test (Ramsey, 1969, *Journal
of the Royal Statistical Society, Series B*) is a widely cited, formal
hypothesis test for functional-form misspecification. It tests whether adding
higher-order powers of the fitted values (here: squared and cubed) to the
model significantly improves fit. A significant result (low p-value) is
evidence that the linear model omits real nonlinear structure. It is a
standard tool in econometrics and applied statistics for exactly this
question, and applies directly here because it only requires a set of fitted
values and a response — it does not care whether those fitted values came from
OLS or from PLS.

## Result on the Concrete Compressive Strength dataset

Running all three diagnostics on a linear PLS baseline (component count chosen
via 10-fold cross-validation) gives a consistent picture:

- The `t1` vs. `y` plot shows clear curvature relative to the linear inner
  relation, most visible at the extremes of the score range.
- The residual-vs-fitted plot shows a non-random, curved pattern rather than
  an even scatter around zero.
- The RESET test **rejects the null hypothesis of correct linear
  specification** (F ≈ 19.4, p ≈ 5×10⁻⁹), i.e. there is strong statistical
  evidence of nonlinearity that the linear PLS model fails to capture.

Together, these three checks — one PLS-specific, one general regression
diagnostic, and one formal statistical test — build a solid, citable case for
moving to a non-linear PLSR approach on this dataset.